# Data Pipeline Walkthrough

This notebook builds the current training input pipeline for the
`yolov8_poly_dist` experiment and inspects it one stage at a time. The pipeline
combines a weighted multi-TFDS detection stream with a separate distance stream,
letterbox pre-resize, mosaic augmentation with per-tile copy-paste, and a parser
that emits fixed-shape training targets in the PolyYOLO radial polygon format.

Pipeline order (training):

```
tfds.load(SkipDecoding) -> repeat -> sample_from_datasets(weights)
  -> shuffle -> decode
  -> letterbox pre-resize to 672x672
  -> zip(copy-paste source) -> padded_batch(group_size)
  -> mosaic (G in -> G//R out, per-tile copy-paste) -> unbatch -> shuffle
  -> parser.parse_fn(is_training=True)   # radial polygon targets, uint8 image
  -> batch(global_batch_size)
  -> zip(distance stream) -> concat on batch dim
  -> prefetch
```

Colour augmentation (normalize / HSV / albumentations) is intentionally **not**
in this graph: parsers emit `uint8`, and colour aug runs per-batch on the
accelerator inside `train_step`. Everything below therefore shows the geometry
and label targets only.

Set the two parameters in the next cell to your local paths, then run top to
bottom. Cells that touch TFDS require the datasets to be built under
`DATA_DIR`; nothing here downloads data.

## Parameters

In [ ]:
# Path to the experiment YAML (authoritative hyperparameter reference).
CONFIG_PATH = "configs/experiments/yolo/yolov8_poly_dist.yaml"

# Local TFDS data directory that holds the built datasets referenced by the YAML
# (cleaner_polygon2026, field_misrecog2026, station_misrecog, cleaner_copy_paste,
# servingbot_polygon). Overrides every tfds_data_dir in the config below.
DATA_DIR = "/home/user/tensorflow_datasets/"

# Run this notebook from the repository root so the package imports resolve.
import os
assert os.path.isdir("data_pipeline"), (
    "Run from the repo root (the directory containing data_pipeline/, configs/, ...)."
)

In [ ]:
import copy
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

from configs.yaml_loader import load_config
from configs.class_map import DETECTION_CLASSES

CLASS_NAMES = [DETECTION_CLASSES[i] for i in sorted(DETECTION_CLASSES)]
print("TensorFlow", tf.__version__, "| classes:", len(CLASS_NAMES))

## Load and retarget the config

The YAML is mapped onto typed dataclasses by a hand-rolled loader. We point every `tfds_data_dir` at `DATA_DIR` so the pipeline reads your local datasets.

In [ ]:
config = load_config(CONFIG_PATH)
task_cfg = config.task
train_cfg = task_cfg.train_data

# Retarget all dataset directories to the local DATA_DIR.
train_cfg.tfds_data_dir = DATA_DIR
task_cfg.validation_data.tfds_data_dir = DATA_DIR
if train_cfg.distance_data is not None:
    train_cfg.distance_data.tfds_data_dir = DATA_DIR

H, W, C = task_cfg.model.input_size
mos = train_cfg.parser.mosaic
print("input_size          :", task_cfg.model.input_size)
print("num_classes         :", task_cfg.num_classes)
print("detection tfds_name :", train_cfg.tfds_name)
print("sampling weights    :", train_cfg.tfds_sampling_weights)
print("copy-paste source   :", train_cfg.tfds_for_cnp, "prob=", train_cfg.prob_copy_n_paste)
print("distance tfds_name  :", train_cfg.distance_data.tfds_name if train_cfg.distance_data else None)
print("detection batch     :", train_cfg.global_batch_size,
      "| distance batch:", train_cfg.distance_data.global_batch_size if train_cfg.distance_data else 0)
print("mosaic              : freq=%.2f group_size=%d decodes_per_output=%d scale=[%.2f, %.2f]"
      % (mos.mosaic_frequency, mos.group_size, mos.decodes_per_output,
         mos.aug_scale_min, mos.aug_scale_max))

## Stage 1 - Decoder output

Each TFDS source is loaded with `SkipDecoding` (images stay JPEG-encoded through
the shuffle buffer) and normalised into a common schema by `PolygonDecoder`. We
run one example through the decoder and inspect the resulting dictionary.

`groundtruth_polygons` is flat interleaved xy (`[N, 3972]` here), input-image
normalised, padded with the reserved `-1.0` sentinel.

In [ ]:
from data_pipeline.tfds_decoders import PolygonDecoder

first_name = train_cfg.tfds_name.split(",")[0].strip()
first_split = train_cfg.tfds_split.split(",")[0].strip()

raw_ds = tfds.load(
    name=first_name,
    split=first_split,
    data_dir=DATA_DIR,
    decoders={"image": tfds.decode.SkipDecoding()},
    read_config=tfds.ReadConfig(assert_cardinality=False),
)
decoder = PolygonDecoder(
    max_vertices=train_cfg.parser.max_vertices,
    num_classes=task_cfg.num_classes,
)
decoded = next(iter(raw_ds.map(decoder.decode).take(1)))

for k, v in decoded.items():
    print(f"{k:22s} {str(v.dtype):>10}  shape={tuple(v.shape)}")
print("\nnum objects:", int(tf.shape(decoded['groundtruth_boxes'])[0]))
print("classes    :", decoded['groundtruth_classes'].numpy().tolist())

## Stage 2 - Letterbox pre-resize

Before grouping, every image is resized to a fixed `672x672` with an
**aspect-preserving letterbox** (long side scaled to 672, gray-114 padding).
Boxes and polygon vertices are transformed through the same scale + pad, and the
`-1.0` polygon sentinel is preserved. This is the single source of truth
(`letterbox_resize`) shared with the eval parser, so training and evaluation see
identical geometry.

In [ ]:
from data_pipeline.augmentations import letterbox_resize

img_lb, boxes_lb, polys_lb = letterbox_resize(
    decoded["image"],
    decoded["groundtruth_boxes"],
    decoded["groundtruth_polygons"],
    H, W,
)
img_lb_np = img_lb.numpy()
boxes_lb_np = boxes_lb.numpy()
print("letterboxed image:", img_lb_np.shape, img_lb_np.dtype)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img_lb_np)
for (y1, x1, y2, x2) in boxes_lb_np:
    ax.add_patch(plt.Rectangle((x1 * W, y1 * H), (x2 - x1) * W, (y2 - y1) * H,
                               fill=False, edgecolor="lime", linewidth=1.5))
ax.set_title("Stage 2: letterboxed 672x672 with transformed GT boxes")
ax.axis("off")
plt.show()

## Stage 3 - Full training stream (mosaic + parser + distance merge)

`build_input_reader_from_config` assembles the entire graph from config: decoder,
letterbox pre-resize, copy-paste module, mosaic module, parser, and the merged
distance stream. One element is `(images, labels)` where `images` is `uint8` and
the batch dimension is the detection batch (128) concatenated with the distance
batch (16) = 144.

The parser targets:

| key            | shape           | meaning                                           |
|----------------|-----------------|---------------------------------------------------|
| `bbox`         | `[B, 300, 4]`   | yxyx normalised, zero-padded past `n_gt`          |
| `classes`      | `[B, 300]`      | class id per object                               |
| `polygons`     | `[B, 300, 72]`  | radial `[dist, angle, conf] x 24` interleaved     |
| `n_gt`         | `[B]`           | valid object count per image                      |
| `ignore_bg`    | `[B]`           | 1 on distance-stream rows (suppresses class loss) |
| `log_distance` | `[B, 300]`      | log-metre distance; `-10.0` sentinel = invalid    |

In [ ]:
from data_pipeline.input_reader import build_input_reader_from_config

train_reader = build_input_reader_from_config(
    data_cfg=train_cfg, task_cfg=task_cfg, is_training=True,
)
train_ds = train_reader()
images, labels = next(iter(train_ds.take(1)))

print("images:", tuple(images.shape), images.dtype, "(uint8, pre colour-aug)")
print("\nlabel tensors:")
for k, v in labels.items():
    print(f"  {k:14s} {str(v.dtype):>8}  shape={tuple(v.shape)}")

## Stage 4 - Mosaic with and without copy-paste

The mosaic stage maps a `group_size` group to `group_size // decodes_per_output`
outputs; each output composites four source tiles onto a 2x canvas, applies one
whole-canvas affine warp, and (per tile, mosaic-only) may paste an RGBA object
crop from the copy-paste source. Non-mosaic single images never receive pastes.

To isolate the effect we build a **detection-only** reader (no distance merge)
twice: once with the copy-paste source enabled and once with it disabled.

In [ ]:
def _detection_only(cfg, with_cnp):
    c = copy.deepcopy(cfg)
    c.distance_data = None          # drop the distance merge for a clean single stream
    if not with_cnp:
        c.tfds_for_cnp = None       # disable copy-paste
    return c

reader_cnp = build_input_reader_from_config(
    data_cfg=_detection_only(train_cfg, with_cnp=True),
    task_cfg=task_cfg, is_training=True)
reader_nocnp = build_input_reader_from_config(
    data_cfg=_detection_only(train_cfg, with_cnp=False),
    task_cfg=task_cfg, is_training=True)

imgs_cnp, _ = next(iter(reader_cnp().take(1)))
imgs_nocnp, _ = next(iter(reader_nocnp().take(1)))

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for j in range(3):
    axes[0, j].imshow(imgs_nocnp[j].numpy()); axes[0, j].axis("off")
    axes[1, j].imshow(imgs_cnp[j].numpy());   axes[1, j].axis("off")
axes[0, 0].set_ylabel("copy-paste OFF", fontsize=12)
axes[1, 0].set_ylabel("copy-paste ON", fontsize=12)
fig.suptitle("Stage 4: mosaic outputs (top: no paste, bottom: per-tile copy-paste)")
plt.tight_layout(); plt.show()

## Stage 5 - Radial polygon target format

Each object's polygon target is `[72]` = 24 bins of `[dist, angle, conf]`
interleaved (15 degree angular steps):

- `dist[i]`  - radial distance of the bin-`i` vertex from the box centre, in
  output-normalised image space.
- `angle[i]` - sub-bin offset in `[0, 1)`; the vertex angle is
  `(i + angle[i]) * (2*pi / 24)`.
- `conf[i]`  - per-bin validity (1 occupied, 0 empty); the decode/visualisation
  gate is `conf >= 0.4`.

Reconstructing cartesian vertices: `x = cx + dist*cos(theta)*W`,
`y = cy + dist*sin(theta)*H`, where `(cx, cy)` is the box centre.

In [ ]:
import math

# Pick the first object of the first image that has a polygon.
b = 0
ng = int(labels["n_gt"][b])
poly = labels["polygons"][b].numpy()          # [300, 72]
box = labels["bbox"][b].numpy()               # [300, 4]

obj = None
for i in range(ng):
    if np.any(poly[i, 2::3] >= 0.5):          # any occupied conf bin
        obj = i
        break

if obj is None:
    print("No polygon object in image 0 of this batch; re-run for another batch.")
else:
    p = poly[obj]
    dist, angle, conf = p[0::3], p[1::3], p[2::3]
    y1, x1, y2, x2 = box[obj]
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    step = 2.0 * math.pi / 24.0
    print(f"object {obj}: class={int(labels['classes'][b][obj])}  centre=({cx:.3f}, {cy:.3f})")
    print(f"occupied bins (conf>=0.4): {int((conf >= 0.4).sum())} / 24")
    print("\n bin   dist   angle   conf")
    for i in range(24):
        mark = "*" if conf[i] >= 0.4 else " "
        print(f"{mark}{i:3d}  {dist[i]:6.3f}  {angle[i]:6.3f}  {conf[i]:5.2f}")

## Stage 6 - Distance-stream merge

The distance dataset (`servingbot_polygon`) is parsed independently, batched to
16, and concatenated onto the detection batch. Those rows carry `ignore_bg = 1`
(class loss suppressed) and real `log_distance` values; detection rows carry the
`-10.0` invalid sentinel.

In [ ]:
ignore_bg = labels["ignore_bg"].numpy()
n_det = int((ignore_bg == 0).sum())
n_dist = int((ignore_bg == 1).sum())
print(f"batch rows: {ignore_bg.shape[0]}  ->  detection={n_det}  distance={n_dist}")

# Valid distances live on the distance rows; -10.0 marks 'no distance'.
log_d = labels["log_distance"].numpy()
valid = log_d > -10.0
if valid.any():
    metres = np.exp(log_d[valid])
    print("valid distance targets: %d  range=[%.2f, %.2f] m  mean=%.2f m"
          % (valid.sum(), metres.min(), metres.max(), metres.mean()))
else:
    print("No valid distance targets in this batch.")

## Final - Training batch visualisation

`common/viz_utils.render_gt_images` renders the post-augmentation GT boxes and
radial polygons exactly as the trainer writes them to TensorBoard. It decodes the
`[72]` radial format the same way as Stage 5. We display the first images of the
merged batch (drawn overlays shown through matplotlib). Requires OpenCV
(`opencv-python-headless`); if unavailable the call returns `None`.

In [ ]:
from common.viz_utils import render_gt_images

k = 6
imgs_f = [images[i].numpy().astype(np.float32) / 255.0 for i in range(k)]
gts = [{
    "bbox":     labels["bbox"][i].numpy(),
    "classes":  labels["classes"][i].numpy(),
    "n_gt":     int(labels["n_gt"][i]),
    "polygons": labels["polygons"][i].numpy(),
} for i in range(k)]

canvas = render_gt_images(imgs_f, gts, draw_box=True, draw_poly=True,
                          class_names=CLASS_NAMES)
if canvas is None:
    print("OpenCV not available - GT overlay skipped.")
else:
    cols = 3
    rows = (k + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    for i, ax in enumerate(np.array(axes).reshape(-1)):
        if i < k:
            ax.imshow(canvas[i]); ax.set_title(f"row {i}  n_gt={gts[i]['n_gt']}")
        ax.axis("off")
    plt.tight_layout(); plt.show()